# DG cross32 + B2-Li 760: длинное обучение
15 эпох по 24 000 примеров, затем 3 полных прохода train на полных кадрах.
DG patch/edge на strides 4/8/16/32, cross-attention на stride 32.
Максимальный RGB размер с шагом 8 для native JPEG до Full HD: 760 (99.358 GFLOPs). 768 превышает лимит (100.377).
При native 1024×1024: 97.553 GFLOPs. Более крупные JPEG требуют отдельного расчёта. Время на H100 ещё не измерено.
Настройте серверные пути и GPU в .env. Для сравнения с предыдущим DG: forward/effective batch 16, accumulation 1 на одной GPU; либо batch 8 на каждой из двух GPU.
Последняя ячейка запускает обучение. resume продолжает только этот run; новый run стартует со стандартных pretrained RGB/JPEG весов.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_long'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
